# Iris データセットの探索と分類モデルの訓練

このノートブックでは、Iris データセットを使った分類問題を TDD で実装します。

## 1. 環境設定とパスの確認

In [49]:
import java.io.File

// 現在のワーキングディレクトリを確認
val currentDir = File(".").absolutePath
println("現在のディレクトリ: $currentDir")

// データファイルのパスを自動検出
val possiblePaths = listOf(
    "src/main/resources/data/iris.csv",           // app/kotlin から実行
    "../src/main/resources/data/iris.csv",        // notebook から実行
    "app/kotlin/src/main/resources/data/iris.csv", // プロジェクトルートから実行
    "../../src/main/resources/data/iris.csv"      // さらに深い場所から実行
)

val dataPath = possiblePaths.firstOrNull { File(it).exists() }
    ?: error("iris.csv が見つかりません。現在のディレクトリ: $currentDir")

println("データファイル: $dataPath")
println("ファイル存在確認: ${File(dataPath).exists()}")

現在のディレクトリ: C:\Users\PC202411-1\IdeaProjects\case-study-game-dev\app\kotlin\notebook\.
データファイル: ../src/main/resources/data/iris.csv
ファイル存在確認: true


## 2. データの読み込みと概要確認

In [50]:
import ml.IrisClassifier

// モデルの作成
val classifier = IrisClassifier(maxDepth = 3)

// データの読み込み
val (X, y) = classifier.loadData(dataPath)

println("=".repeat(60))
println("データの概要")
println("=".repeat(60))
println("サンプル数: ${X.size}")
println("特徴量数: ${X[0].size}")
println("クラス数: ${y.distinct().size}")
println()

// データの最初の5行を表示
println("最初の5サンプル:")
println("sepal_length, sepal_width, petal_length, petal_width, species")
for (i in 0 until minOf(5, X.size)) {
    println("${X[i][0]}, ${X[i][1]}, ${X[i][2]}, ${X[i][3]}, ${y[i]}")
}

データの概要
サンプル数: 143
特徴量数: 4
クラス数: 3

最初の5サンプル:
sepal_length, sepal_width, petal_length, petal_width, species
0.22, 0.63, 0.08, 0.04, Iris-setosa
0.17, 0.42, 0.35, 0.04, Iris-setosa
0.11, 0.5, 0.13, 0.04, Iris-setosa
0.08, 0.46, 0.26, 0.04, Iris-setosa
0.19, 0.67, 0.44, 0.04, Iris-setosa


## 3. Lets-Plot の初期化

In [51]:
%use lets-plot

## 4. データの可視化

### 4.1 特徴量の分布（ヒストグラム）- sepal_length

In [69]:
val data1 = mapOf(
    "value" to X.map { it[0] },
    "species" to y.toList()
)

letsPlot(data1) +
    geomHistogram(alpha = 0.6, bins = 20) {
        x = "value"
        fill = "species"
    } +
    scaleFillManual(values = listOf("#FF6B6B", "#4ECDC4", "#45B7D1")) +
    ggtitle("sepal_length の分布") +
    labs(x = "sepal_length", y = "度数")

The problem is found in one of the loaded libraries: check library renderers
java.lang.NoSuchMethodError: 'java.util.Map org.jetbrains.letsPlot.intern.ToSpecConvertersKt.toSpec(org.jetbrains.letsPlot.Figure)'
org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryException: The problem is found in one of the loaded libraries: check library renderers
	at org.jetbrains.kotlinx.jupyter.exceptions.CompositeReplExceptionKt.throwAsLibraryException(CompositeReplException.kt:48)
	at org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryExceptionKt.rethrowAsLibraryException(ReplLibraryException.kt:42)
	at org.jetbrains.kotlinx.jupyter.codegen.RenderersProcessorImpl.renderResult(RenderersProcessorImpl.kt:35)
	at org.jetbrains.kotlinx.jupyter.repl.impl.ReplForJupyterImpl.renderResult$lambda$0$0(ReplForJupyterImpl.kt:536)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll(Logging.kt:33)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll$default(Logging.kt:27)
	at org.jetbrains.kotlinx

null

### 4.2 特徴量の分布（ヒストグラム）- petal_length

In [53]:
val data2 = mapOf(
    "value" to X.map { it[2] },
    "species" to y.toList()
)

letsPlot(data2) +
    geomHistogram(alpha = 0.6, bins = 20) {
        x = "value"
        fill = "species"
    } +
    scaleFillManual(values = listOf("#FF6B6B", "#4ECDC4", "#45B7D1")) +
    ggtitle("petal_length の分布") +
    labs(x = "petal_length", y = "度数")

The problem is found in one of the loaded libraries: check library renderers
java.lang.NoSuchMethodError: 'java.util.Map org.jetbrains.letsPlot.intern.ToSpecConvertersKt.toSpec(org.jetbrains.letsPlot.Figure)'
org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryException: The problem is found in one of the loaded libraries: check library renderers
	at org.jetbrains.kotlinx.jupyter.exceptions.CompositeReplExceptionKt.throwAsLibraryException(CompositeReplException.kt:48)
	at org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryExceptionKt.rethrowAsLibraryException(ReplLibraryException.kt:42)
	at org.jetbrains.kotlinx.jupyter.codegen.RenderersProcessorImpl.renderResult(RenderersProcessorImpl.kt:35)
	at org.jetbrains.kotlinx.jupyter.repl.impl.ReplForJupyterImpl.renderResult$lambda$0$0(ReplForJupyterImpl.kt:536)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll(Logging.kt:33)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll$default(Logging.kt:27)
	at org.jetbrains.kotlinx

null

### 4.3 散布図 - 花びらの長さ vs 幅

In [54]:
val petalData = mapOf(
    "petal_length" to X.map { it[2] },
    "petal_width" to X.map { it[3] },
    "species" to y.toList()
)

letsPlot(petalData) +
    geomPoint(size = 3, alpha = 0.7) {
        x = "petal_length"
        y = "petal_width"
        color = "species"
    } +
    scaleColorManual(values = listOf("#FF6B6B", "#4ECDC4", "#45B7D1")) +
    ggtitle("花びらの長さ vs 幅") +
    labs(x = "petal_length (cm)", y = "petal_width (cm)")

The problem is found in one of the loaded libraries: check library renderers
java.lang.NoSuchMethodError: 'java.util.Map org.jetbrains.letsPlot.intern.ToSpecConvertersKt.toSpec(org.jetbrains.letsPlot.Figure)'
org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryException: The problem is found in one of the loaded libraries: check library renderers
	at org.jetbrains.kotlinx.jupyter.exceptions.CompositeReplExceptionKt.throwAsLibraryException(CompositeReplException.kt:48)
	at org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryExceptionKt.rethrowAsLibraryException(ReplLibraryException.kt:42)
	at org.jetbrains.kotlinx.jupyter.codegen.RenderersProcessorImpl.renderResult(RenderersProcessorImpl.kt:35)
	at org.jetbrains.kotlinx.jupyter.repl.impl.ReplForJupyterImpl.renderResult$lambda$0$0(ReplForJupyterImpl.kt:536)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll(Logging.kt:33)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll$default(Logging.kt:27)
	at org.jetbrains.kotlinx

null

### 4.4 散布図 - がく片の長さ vs 幅

In [55]:
val sepalData = mapOf(
    "sepal_length" to X.map { it[0] },
    "sepal_width" to X.map { it[1] },
    "species" to y.toList()
)

letsPlot(sepalData) +
    geomPoint(size = 3, alpha = 0.7) {
        x = "sepal_length"
        y = "sepal_width"
        color = "species"
    } +
    scaleColorManual(values = listOf("#FF6B6B", "#4ECDC4", "#45B7D1")) +
    ggtitle("がく片の長さ vs 幅") +
    labs(x = "sepal_length (cm)", y = "sepal_width (cm)")

The problem is found in one of the loaded libraries: check library renderers
java.lang.NoSuchMethodError: 'java.util.Map org.jetbrains.letsPlot.intern.ToSpecConvertersKt.toSpec(org.jetbrains.letsPlot.Figure)'
org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryException: The problem is found in one of the loaded libraries: check library renderers
	at org.jetbrains.kotlinx.jupyter.exceptions.CompositeReplExceptionKt.throwAsLibraryException(CompositeReplException.kt:48)
	at org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryExceptionKt.rethrowAsLibraryException(ReplLibraryException.kt:42)
	at org.jetbrains.kotlinx.jupyter.codegen.RenderersProcessorImpl.renderResult(RenderersProcessorImpl.kt:35)
	at org.jetbrains.kotlinx.jupyter.repl.impl.ReplForJupyterImpl.renderResult$lambda$0$0(ReplForJupyterImpl.kt:536)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll(Logging.kt:33)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll$default(Logging.kt:27)
	at org.jetbrains.kotlinx

null

## 5. 基本統計量の確認

In [56]:
val featureNames = listOf("sepal_length", "sepal_width", "petal_length", "petal_width")

println("基本統計量:")
println("-".repeat(60))

featureNames.forEachIndexed { idx, name ->
    val values = X.map { it[idx] }
    val min = values.minOrNull() ?: 0.0
    val max = values.maxOrNull() ?: 0.0
    val mean = values.average()
    val sorted = values.sorted()
    val median = if (sorted.size % 2 == 0) {
        (sorted[sorted.size / 2 - 1] + sorted[sorted.size / 2]) / 2.0
    } else {
        sorted[sorted.size / 2]
    }
    
    println("$name:")
    println("  最小値: %.2f".format(min))
    println("  最大値: %.2f".format(max))
    println("  平均値: %.2f".format(mean))
    println("  中央値: %.2f".format(median))
    println()
}

基本統計量:
------------------------------------------------------------
sepal_length:
  最小値: 0.00
  最大値: 0.94
  平均値: 0.41
  中央値: 0.39

sepal_width:
  最小値: 0.00
  最大値: 1.00
  平均値: 0.44
  中央値: 0.42

petal_length:
  最小値: 0.01
  最大値: 0.95
  平均値: 0.48
  中央値: 0.47

petal_width:
  最小値: 0.01
  最大値: 0.96
  平均値: 0.44
  中央値: 0.50



## 6. クラスの分布

In [57]:
println("クラスの分布:")
println("-".repeat(60))

val total = y.size.toDouble()
val classCounts = y.groupBy { it }.mapValues { it.value.size }

classCounts.toSortedMap().forEach { (species, count) ->
    val percentage = (count / total * 100)
    println("$species: $count 件 (%.1f%%)".format(percentage))
}
println()

クラスの分布:
------------------------------------------------------------
Iris-setosa: 50 件 (35.0%)
Iris-versicolor: 48 件 (33.6%)
Iris-virginica: 45 件 (31.5%)



### クラス分布の棒グラフ

In [58]:
val barData = mapOf(
    "species" to classCounts.keys.toList(),
    "count" to classCounts.values.toList()
)

letsPlot(barData) +
    geomBar(stat = Stat.identity, alpha = 0.8) {
        x = "species"
        y = "count"
        fill = "species"
    } +
    scaleFillManual(values = listOf("#FF6B6B", "#4ECDC4", "#45B7D1")) +
    ggtitle("クラスの分布") +
    labs(x = "Species", y = "サンプル数")

The problem is found in one of the loaded libraries: check library renderers
java.lang.NoSuchMethodError: 'java.util.Map org.jetbrains.letsPlot.intern.ToSpecConvertersKt.toSpec(org.jetbrains.letsPlot.Figure)'
org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryException: The problem is found in one of the loaded libraries: check library renderers
	at org.jetbrains.kotlinx.jupyter.exceptions.CompositeReplExceptionKt.throwAsLibraryException(CompositeReplException.kt:48)
	at org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryExceptionKt.rethrowAsLibraryException(ReplLibraryException.kt:42)
	at org.jetbrains.kotlinx.jupyter.codegen.RenderersProcessorImpl.renderResult(RenderersProcessorImpl.kt:35)
	at org.jetbrains.kotlinx.jupyter.repl.impl.ReplForJupyterImpl.renderResult$lambda$0$0(ReplForJupyterImpl.kt:536)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll(Logging.kt:33)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll$default(Logging.kt:27)
	at org.jetbrains.kotlinx

null

## 7. モデルの訓練

In [59]:
println("モデルの訓練中...")
val startTime = System.currentTimeMillis()
classifier.train(X, y)
val trainingTime = System.currentTimeMillis() - startTime

println("訓練完了！")
println("訓練時間: ${trainingTime}ms")
println()

モデルの訓練中...
訓練完了！
訓練時間: 1ms



## 8. モデルの評価

In [60]:
val trainAccuracy = classifier.evaluate(X, y)
println("=".repeat(60))
println("モデルの評価結果")
println("=".repeat(60))
println("訓練正解率: %.2f%%".format(trainAccuracy * 100))
println()

モデルの評価結果
訓練正解率: 95.10%



## 9. 混同行列

In [61]:
println("混同行列:")
val predictions = classifier.predict(X)
val species = y.distinct().sorted()

println("実際 \\ 予測 | " + species.joinToString(" | "))
println("-".repeat(60))

// 混同行列のデータを準備
val confusionMatrix = mutableListOf<Triple<String, String, Int>>()

species.forEach { actualSpecies ->
    val actualIndices = y.indices.filter { y[it] == actualSpecies }
    print("%-15s | ".format(actualSpecies))
    species.forEach { predSpecies ->
        val count = actualIndices.count { predictions[it] == predSpecies }
        print("%3d | ".format(count))
        confusionMatrix.add(Triple(actualSpecies, predSpecies, count))
    }
    println()
}
println()

混同行列:
実際 \ 予測 | Iris-setosa | Iris-versicolor | Iris-virginica
------------------------------------------------------------
Iris-setosa     |  50 |   0 |   0 | 
Iris-versicolor |   0 |  47 |   1 | 
Iris-virginica  |   0 |   6 |  39 | 



### 混同行列のヒートマップ

In [62]:
val heatmapData = mapOf(
    "actual" to confusionMatrix.map { it.first },
    "predicted" to confusionMatrix.map { it.second },
    "count" to confusionMatrix.map { it.third }
)

letsPlot(heatmapData) +
    geomTile(alpha = 0.9) {
        x = "predicted"
        y = "actual"
        fill = "count"
    } +
    geomText(color = "white", size = 12) {
        x = "predicted"
        y = "actual"
        label = "count"
    } +
    ggtitle("混同行列") +
    labs(x = "予測されたクラス", y = "実際のクラス")

The problem is found in one of the loaded libraries: check library renderers
java.lang.NoSuchMethodError: 'java.util.Map org.jetbrains.letsPlot.intern.ToSpecConvertersKt.toSpec(org.jetbrains.letsPlot.Figure)'
org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryException: The problem is found in one of the loaded libraries: check library renderers
	at org.jetbrains.kotlinx.jupyter.exceptions.CompositeReplExceptionKt.throwAsLibraryException(CompositeReplException.kt:48)
	at org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryExceptionKt.rethrowAsLibraryException(ReplLibraryException.kt:42)
	at org.jetbrains.kotlinx.jupyter.codegen.RenderersProcessorImpl.renderResult(RenderersProcessorImpl.kt:35)
	at org.jetbrains.kotlinx.jupyter.repl.impl.ReplForJupyterImpl.renderResult$lambda$0$0(ReplForJupyterImpl.kt:536)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll(Logging.kt:33)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll$default(Logging.kt:27)
	at org.jetbrains.kotlinx

null

## 10. クラス別の性能

In [63]:
println("クラス別の性能:")
println("-".repeat(60))

val classAccuracies = mutableListOf<Pair<String, Double>>()

species.forEach { targetSpecies ->
    val indices = y.indices.filter { y[it] == targetSpecies }
    val correct = indices.count { predictions[it] == targetSpecies }
    val total = indices.size
    val classAccuracy = correct.toDouble() / total
    classAccuracies.add(Pair(targetSpecies, classAccuracy))

    println("$targetSpecies:")
    println("  正解率: %.2f%% ($correct / $total)".format(classAccuracy * 100))
}
println()

クラス別の性能:
------------------------------------------------------------
Iris-setosa:
  正解率: 100.00% (50 / 50)
Iris-versicolor:
  正解率: 97.92% (47 / 48)
Iris-virginica:
  正解率: 86.67% (39 / 45)



### クラス別正解率の棒グラフ

In [64]:
val accuracyData = mapOf(
    "species" to classAccuracies.map { it.first },
    "accuracy" to classAccuracies.map { it.second * 100 }
)

letsPlot(accuracyData) +
    geomBar(stat = Stat.identity, alpha = 0.8) {
        x = "species"
        y = "accuracy"
        fill = "species"
    } +
    scaleFillManual(values = listOf("#FF6B6B", "#4ECDC4", "#45B7D1")) +
    ggtitle("クラス別の正解率") +
    labs(x = "Species", y = "正解率 (%)")

The problem is found in one of the loaded libraries: check library renderers
java.lang.NoSuchMethodError: 'java.util.Map org.jetbrains.letsPlot.intern.ToSpecConvertersKt.toSpec(org.jetbrains.letsPlot.Figure)'
org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryException: The problem is found in one of the loaded libraries: check library renderers
	at org.jetbrains.kotlinx.jupyter.exceptions.CompositeReplExceptionKt.throwAsLibraryException(CompositeReplException.kt:48)
	at org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryExceptionKt.rethrowAsLibraryException(ReplLibraryException.kt:42)
	at org.jetbrains.kotlinx.jupyter.codegen.RenderersProcessorImpl.renderResult(RenderersProcessorImpl.kt:35)
	at org.jetbrains.kotlinx.jupyter.repl.impl.ReplForJupyterImpl.renderResult$lambda$0$0(ReplForJupyterImpl.kt:536)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll(Logging.kt:33)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll$default(Logging.kt:27)
	at org.jetbrains.kotlinx

null

## 11. 個別予測の例

In [65]:
println("個別予測の例:")
println("-".repeat(60))

val testSamples = listOf(
    Triple(doubleArrayOf(5.1, 3.5, 1.4, 0.2), "setosa", "setosa の典型的な特徴"),
    Triple(doubleArrayOf(6.5, 3.0, 5.2, 2.0), "virginica", "virginica の典型的な特徴"),
    Triple(doubleArrayOf(5.7, 2.8, 4.1, 1.3), "versicolor", "versicolor の典型的な特徴")
)

testSamples.forEachIndexed { i, (sample, expected, description) ->
    val prediction = classifier.predict(arrayOf(sample))[0]
    val isCorrect = prediction == expected
    val mark = if (isCorrect) "✓" else "✗"
    println("$mark サンプル ${i+1} ($description):")
    println("  特徴量: [${sample.joinToString(", ")}]")
    println("  予測: $prediction (期待: $expected)")
    println()
}

個別予測の例:
------------------------------------------------------------
✗ サンプル 1 (setosa の典型的な特徴):
  特徴量: [5.1, 3.5, 1.4, 0.2]
  予測: Iris-setosa (期待: setosa)

✗ サンプル 2 (virginica の典型的な特徴):
  特徴量: [6.5, 3.0, 5.2, 2.0]
  予測: Iris-virginica (期待: virginica)

✗ サンプル 3 (versicolor の典型的な特徴):
  特徴量: [5.7, 2.8, 4.1, 1.3]
  予測: Iris-virginica (期待: versicolor)



### 予測結果の可視化

In [66]:
// 元のデータと新しいサンプルを一緒にプロット
val allPetalLength = X.map { it[2] }.toMutableList()
val allPetalWidth = X.map { it[3] }.toMutableList()
val allSpecies = y.toMutableList()
val allType = MutableList(X.size) { "元データ" }

// 新しいサンプルを追加
testSamples.forEach { (sample, expected, _) ->
    allPetalLength.add(sample[2])
    allPetalWidth.add(sample[3])
    val prediction = classifier.predict(arrayOf(sample))[0]
    allSpecies.add(prediction)
    allType.add("予測")
}

val combinedData = mapOf(
    "petal_length" to allPetalLength,
    "petal_width" to allPetalWidth,
    "species" to allSpecies,
    "type" to allType
)

letsPlot(combinedData) +
    geomPoint(alpha = 0.7) {
        x = "petal_length"
        y = "petal_width"
        color = "species"
        size = "type"
        shape = "type"
    } +
    scaleColorManual(values = listOf("#FF6B6B", "#4ECDC4", "#45B7D1")) +
    ggtitle("予測結果の可視化") +
    labs(x = "petal_length (cm)", y = "petal_width (cm)")

The problem is found in one of the loaded libraries: check library renderers
java.lang.NoSuchMethodError: 'java.util.Map org.jetbrains.letsPlot.intern.ToSpecConvertersKt.toSpec(org.jetbrains.letsPlot.Figure)'
org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryException: The problem is found in one of the loaded libraries: check library renderers
	at org.jetbrains.kotlinx.jupyter.exceptions.CompositeReplExceptionKt.throwAsLibraryException(CompositeReplException.kt:48)
	at org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryExceptionKt.rethrowAsLibraryException(ReplLibraryException.kt:42)
	at org.jetbrains.kotlinx.jupyter.codegen.RenderersProcessorImpl.renderResult(RenderersProcessorImpl.kt:35)
	at org.jetbrains.kotlinx.jupyter.repl.impl.ReplForJupyterImpl.renderResult$lambda$0$0(ReplForJupyterImpl.kt:536)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll(Logging.kt:33)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll$default(Logging.kt:27)
	at org.jetbrains.kotlinx

null

## 12. モデルの保存

In [67]:
import java.io.File

// モデル保存先のパスを構築
val modelDir = when {
    File("model").exists() || File(".").resolve("model").parentFile.exists() -> "model"
    File("../model").parentFile.exists() -> "../model"
    File("app/kotlin/model").parentFile.exists() -> "app/kotlin/model"
    else -> "model" // デフォルト
}

val modelPath = "$modelDir/iris_model.ser"
File(modelPath).parentFile?.mkdirs()

classifier.saveModel(modelPath)

println("=".repeat(60))
println("モデルの保存完了")
println("=".repeat(60))
println("保存先: $modelPath")
println("ファイルサイズ: ${File(modelPath).length()} bytes")

モデルの保存完了
保存先: model/iris_model.ser
ファイルサイズ: 2142 bytes


## まとめ

このノートブックでは、Iris データセットを使った分類問題を TDD で実装しました。

### 実施内容

1. **データの読み込みと探索**: CSV からのデータ読み込み、欠損値の処理
2. **データの可視化**: ヒストグラム、散布図、棒グラフによる分布確認
3. **モデルの訓練**: 決定木モデルの訓練（max_depth=3）
4. **モデルの評価**: 正解率の計算、混同行列の作成
5. **予測の実行**: 新しいデータでの分類予測と可視化
6. **モデルの保存**: モデルの永続化

### 次のステップ

- データを訓練用とテスト用に分割して過学習をチェック
- クロスバリデーションで性能を評価
- 他のデータセット（Cinema、Boston、Survived）に挑戦
- Web API 化（Ktor による REST API 実装）

---

お疲れ様でした！TDD による機械学習開発の基礎を習得しました！